In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/daisee_train_visual_features_rollfix.csv
/kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/mindbridge_behavior_model.json
/kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/daisee_validation_visual_features.csv
/kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/behavior_model_info.json
/kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/daisee_test_visual_features.csv
/kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/mindbridge_visual_model.json
/kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/visual_model_info.json


In [2]:
import pandas as pd
import numpy as np
import glob

base = glob.glob(
    "/kaggle/input/**/daisee_train_visual_features_rollfix.csv",
    recursive=True
)[0]

train_path = base

val_path = glob.glob(
    "/kaggle/input/**/daisee_validation_visual_features.csv",
    recursive=True
)[0]

test_path = glob.glob(
    "/kaggle/input/**/daisee_test_visual_features.csv",
    recursive=True
)[0]

train_visual = pd.read_csv(train_path)
val_visual = pd.read_csv(val_path)
test_visual = pd.read_csv(test_path)

print("Train:", train_visual.shape)
print("Validation:", val_visual.shape)
print("Test:", test_visual.shape)

Train: (5358, 16)
Validation: (1429, 16)
Test: (1784, 16)


In [3]:
# =========================================================
# VISUAL RELIABILITY v1
# =========================================================

pose_cols = [
    "mean_abs_pitch",
    "mean_abs_yaw",
    "mean_abs_roll"
]

# -----------------------------------------
# 1. Learn normal pose ranges from TRAIN only
# -----------------------------------------

pose_thresholds = {}

for col in pose_cols:

    q75 = train_visual[col].quantile(0.75)
    q95 = train_visual[col].quantile(0.95)

    pose_thresholds[col] = {
        "q75": q75,
        "q95": q95
    }

print("TRAIN-derived pose thresholds:")

for col, values in pose_thresholds.items():

    print(
        col,
        "| q75 =",
        round(values["q75"], 3),
        "| q95 =",
        round(values["q95"], 3)
    )


# -----------------------------------------
# 2. Reliability function
# -----------------------------------------

def add_visual_reliability(df):

    result = df.copy()

    pose_quality_parts = []

    for col in pose_cols:

        q75 = pose_thresholds[col]["q75"]
        q95 = pose_thresholds[col]["q95"]

        # Below q75 -> penalty 0
        # Between q75-q95 -> gradually increases
        # Above q95 -> penalty 1
        penalty = (
            (result[col] - q75) /
            (q95 - q75 + 1e-8)
        )

        penalty = np.clip(
            penalty,
            0,
            1
        )

        pose_quality_parts.append(
            1 - penalty
        )

    # Average pitch/yaw/roll quality
    result["pose_quality"] = np.mean(
        pose_quality_parts,
        axis=0
    )

    # Final reliability:
    # visibility is primary
    # pose quality reduces trust when face angle is extreme
    result["visual_reliability"] = (
        result["face_visibility_ratio"]
        *
        (
            0.5 +
            0.5 * result["pose_quality"]
        )
    )

    result["visual_reliability"] = np.clip(
        result["visual_reliability"],
        0,
        1
    )

    return result


# -----------------------------------------
# 3. Apply same rule to all splits
# -----------------------------------------

train_visual_rel = add_visual_reliability(
    train_visual
)

val_visual_rel = add_visual_reliability(
    val_visual
)

test_visual_rel = add_visual_reliability(
    test_visual
)

print("\nVisual reliability created ✅")

TRAIN-derived pose thresholds:
mean_abs_pitch | q75 = 10.601 | q95 = 18.192
mean_abs_yaw | q75 = 11.366 | q95 = 20.608
mean_abs_roll | q75 = 6.369 | q95 = 11.243

Visual reliability created ✅


In [4]:
for name, df in [
    ("Train", train_visual_rel),
    ("Validation", val_visual_rel),
    ("Test", test_visual_rel)
]:

    print(f"\n=== {name} ===")

    print(
        df["visual_reliability"]
        .describe()
    )

    print(
        "Reliability < 0.8:",
        (
            df["visual_reliability"] < 0.8
        ).sum()
    )

    print(
        "Reliability < 0.6:",
        (
            df["visual_reliability"] < 0.6
        ).sum()
    )


=== Train ===
count    5358.000000
mean        0.938349
std         0.082258
min         0.479895
25%         0.887678
50%         0.984701
75%         1.000000
max         1.000000
Name: visual_reliability, dtype: float64
Reliability < 0.8: 367
Reliability < 0.6: 4

=== Validation ===
count    1429.000000
mean        0.915530
std         0.099225
min         0.521562
25%         0.833333
50%         0.957961
75%         1.000000
max         1.000000
Name: visual_reliability, dtype: float64
Reliability < 0.8: 174
Reliability < 0.6: 7

=== Test ===
count    1784.000000
mean        0.915424
std         0.112625
min         0.500000
25%         0.838230
50%         0.982184
75%         1.000000
max         1.000000
Name: visual_reliability, dtype: float64
Reliability < 0.8: 317
Reliability < 0.6: 13


In [5]:
import glob
from xgboost import XGBClassifier

visual_model_path = glob.glob(
    "/kaggle/input/**/mindbridge_visual_model.json",
    recursive=True
)[0]

visual_model = XGBClassifier()
visual_model.load_model(visual_model_path)

print("Visual model loaded ✅")
print("Path:", visual_model_path)

Visual model loaded ✅
Path: /kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/mindbridge_visual_model.json


In [6]:
visual_features = [
    "mean_eye_openness",
    "std_eye_openness",
    "mean_gaze_ratio",
    "std_gaze_ratio",
    "mean_pitch",
    "mean_yaw",
    "mean_roll",
    "mean_abs_pitch",
    "mean_abs_yaw",
    "mean_abs_roll",
    "face_visibility_ratio"
]


def add_visual_evidence(df):

    result = df.copy()

    # Raw confusion probability from visual model
    result["visual_confusion_prob"] = (
        visual_model.predict_proba(
            result[visual_features]
        )[:, 1]
    )

    # Reliability-aware adjustment:
    # low reliability -> probability moves toward neutral 0.5
    result["visual_confusion_evidence"] = (
        0.5
        +
        result["visual_reliability"]
        *
        (
            result["visual_confusion_prob"]
            - 0.5
        )
    )

    return result


train_visual_rel = add_visual_evidence(
    train_visual_rel
)

val_visual_rel = add_visual_evidence(
    val_visual_rel
)

test_visual_rel = add_visual_evidence(
    test_visual_rel
)

print("Visual evidence created ✅")

Visual evidence created ✅


In [7]:
print(
    train_visual_rel[
        [
            "ClipID",
            "visual_confusion_prob",
            "visual_reliability",
            "visual_confusion_evidence"
        ]
    ].head(10)
)

print("\nVisual evidence statistics:")

print(
    train_visual_rel[
        [
            "visual_confusion_prob",
            "visual_reliability",
            "visual_confusion_evidence"
        ]
    ].describe()
)

           ClipID  visual_confusion_prob  visual_reliability  \
0  1100011002.avi               0.518518            0.947454   
1  1100011003.avi               0.431598            0.994663   
2  1100011004.avi               0.419003            0.996906   
3  1100011005.avi               0.376906            0.980325   
4  1100011006.avi               0.420519            0.960775   
5  1100011007.avi               0.369988            0.931725   
6  1100011008.avi               0.255538            0.915157   
7  1100011009.avi               0.349156            0.934595   
8  1100011010.avi               0.331347            0.955040   
9  1100011011.avi               0.456415            0.970251   

   visual_confusion_evidence  
0                   0.517545  
1                   0.431963  
2                   0.419253  
3                   0.379328  
4                   0.423637  
5                   0.378865  
6                   0.276279  
7                   0.359022  
8               

In [8]:
import json
import glob

behavior_info_path = glob.glob(
    "/kaggle/input/**/behavior_model_info.json",
    recursive=True
)[0]

with open(behavior_info_path, "r") as f:
    behavior_info = json.load(f)

print("=== BEHAVIOUR MODEL INFO ===")

for key, value in behavior_info.items():
    print(f"\n{key}:")
    print(value)

=== BEHAVIOUR MODEL INFO ===

model:
XGBoost Behaviour Baseline

target:
Confusion

target_rule:
Average_confidence(CONFUSED) >= 0.5

threshold:
0.79

features:
['correct', 'bottom_hint', 'hint_count', 'attempt_count', 'ms_first_response', 'overlap_time', 'first_action', 'position', 'original', 'bottom_hint_missing']

dataset:
ASSISTments 2012-2013 with Affect

note:
Public-dataset baseline model for MindBridge research


In [11]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "2012-2013" in file:
            print(os.path.join(root, file))

/kaggle/input/datasets/rasangabawanthi/mindbridge-assistments-2012-2013/2012-2013-data-with-predictions-4-final.csv


In [12]:
import pandas as pd
import numpy as np

behavior_path = "/kaggle/input/datasets/rasangabawanthi/mindbridge-assistments-2012-2013/2012-2013-data-with-predictions-4-final.csv"

behavior_cols = [
    "user_id",
    "start_time",

    "correct",
    "bottom_hint",
    "hint_count",
    "attempt_count",
    "ms_first_response",
    "overlap_time",
    "first_action",
    "position",
    "original",

    "Average_confidence(CONFUSED)"
]

behavior_df = pd.read_csv(
    behavior_path,
    usecols=behavior_cols,
    low_memory=False
)

print("Shape:", behavior_df.shape)
print("\nColumns:")
print(behavior_df.columns.tolist())

print("\nMemory usage (GB):")
print(
    round(
        behavior_df.memory_usage(
            deep=True
        ).sum() / 1024**3,
        3
    )
)

print("\nMissing values:")
print(
    behavior_df.isnull().sum()
)

Shape: (6123270, 12)

Columns:
['user_id', 'start_time', 'original', 'correct', 'bottom_hint', 'hint_count', 'attempt_count', 'ms_first_response', 'position', 'overlap_time', 'first_action', 'Average_confidence(CONFUSED)']

Memory usage (GB):
0.89

Missing values:
user_id                             0
start_time                          0
original                            0
correct                             0
bottom_hint                     60348
hint_count                          0
attempt_count                       0
ms_first_response                   0
position                            0
overlap_time                        0
first_action                        0
Average_confidence(CONFUSED)        0
dtype: int64


In [13]:
# =========================================================
# BEHAVIOUR RELIABILITY - DATA QUALITY INSPECTION
# =========================================================

# Missing indicator used by the trained behaviour model
behavior_df["bottom_hint_missing"] = (
    behavior_df["bottom_hint"].isna()
).astype(int)

# Basic invalid-value checks
print("=== INVALID / SUSPICIOUS VALUE COUNTS ===")

checks = {
    "ms_first_response < 0":
        (behavior_df["ms_first_response"] < 0).sum(),

    "overlap_time < 0":
        (behavior_df["overlap_time"] < 0).sum(),

    "attempt_count < 0":
        (behavior_df["attempt_count"] < 0).sum(),

    "hint_count < 0":
        (behavior_df["hint_count"] < 0).sum(),

    "position < 0":
        (behavior_df["position"] < 0).sum(),

    "bottom_hint missing":
        behavior_df["bottom_hint_missing"].sum()
}

for name, count in checks.items():
    print(f"{name}: {count}")


# ---------------------------------------------------------
# Empirical distributions
# ---------------------------------------------------------

inspect_cols = [
    "ms_first_response",
    "overlap_time",
    "attempt_count",
    "hint_count",
    "position"
]

quantiles = behavior_df[
    inspect_cols
].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

print("\n=== FEATURE QUANTILES ===")
quantiles

=== INVALID / SUSPICIOUS VALUE COUNTS ===
ms_first_response < 0: 13
overlap_time < 0: 13
attempt_count < 0: 0
hint_count < 0: 0
position < 0: 0
bottom_hint missing: 60348

=== FEATURE QUANTILES ===


,0.01,0.05,0.25,0.50,0.75,0.95,0.99
ms_first_response,1969.0,3608.0,9436.0,22331.0,54865.0,182432.0,377531.31
overlap_time,1977.0,3621.0,9468.0,22415.0,55054.0,183100.0,378986.93
attempt_count,1.0,1.0,1.0,1.0,1.0,3.0,5.00
hint_count,0.0,0.0,0.0,0.0,0.0,3.0,4.00
position,0.0,1.0,4.0,12.0,38.0,167.0,1662.00


In [14]:
print("Response-time quantiles in seconds:")

for q in [0.01, 0.05, 0.50, 0.95, 0.99]:

    value_ms = behavior_df[
        "ms_first_response"
    ].quantile(q)

    print(
        f"{int(q*100)}th percentile:",
        round(value_ms / 1000, 2),
        "seconds"
    )

Response-time quantiles in seconds:
1th percentile: 1.97 seconds
5th percentile: 3.61 seconds
50th percentile: 22.33 seconds
95th percentile: 182.43 seconds
99th percentile: 377.53 seconds


In [15]:
# =========================================================
# BEHAVIOUR RELIABILITY v1
# Data-quality / observability based
# =========================================================

def add_behavior_reliability(df):

    result = df.copy()

    # Start with perfect reliability
    reliability = np.ones(len(result), dtype=float)

    # -------------------------------------------------
    # 1. Missing bottom_hint
    # Small penalty because the rest of behaviour data
    # is still available
    # -------------------------------------------------

    reliability -= (
        result["bottom_hint_missing"].values
        * 0.10
    )

    # -------------------------------------------------
    # 2. Invalid negative timing values
    # Strong penalty because timing evidence is unreliable
    # -------------------------------------------------

    invalid_time = (
        (result["ms_first_response"] < 0)
        |
        (result["overlap_time"] < 0)
    )

    reliability -= (
        invalid_time.astype(float).values
        * 0.50
    )

    # -------------------------------------------------
    # 3. Extremely short response time
    # Below 1st percentile (~1.97 sec)
    # Small penalty only
    # -------------------------------------------------

    very_short = (
        result["ms_first_response"] < 1969
    )

    reliability -= (
        very_short.astype(float).values
        * 0.10
    )

    # -------------------------------------------------
    # 4. Extremely long response time
    # Above 99th percentile (~377 sec)
    # Treat cautiously; could be idle/logging anomaly
    # -------------------------------------------------

    very_long = (
        result["ms_first_response"] > 377531.31
    )

    reliability -= (
        very_long.astype(float).values
        * 0.10
    )

    # Keep reliability in [0, 1]
    result["behavior_reliability"] = np.clip(
        reliability,
        0,
        1
    )

    return result


behavior_rel_df = add_behavior_reliability(
    behavior_df
)

print("Behaviour Reliability v1 created ✅")

Behaviour Reliability v1 created ✅


In [16]:
print(
    behavior_rel_df[
        "behavior_reliability"
    ].describe()
)

print(
    "\nReliability < 1.0:",
    (
        behavior_rel_df[
            "behavior_reliability"
        ] < 1.0
    ).sum()
)

print(
    "Reliability < 0.9:",
    (
        behavior_rel_df[
            "behavior_reliability"
        ] < 0.9
    ).sum()
)

print(
    "Reliability < 0.6:",
    (
        behavior_rel_df[
            "behavior_reliability"
        ] < 0.6
    ).sum()
)

print("\nLowest reliability rows:")

print(
    behavior_rel_df[
        [
            "user_id",
            "ms_first_response",
            "overlap_time",
            "bottom_hint_missing",
            "behavior_reliability"
        ]
    ]
    .sort_values(
        "behavior_reliability"
    )
    .head(20)
)

count    6.123270e+06
mean     9.970147e-01
std      1.715499e-02
min      4.000000e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: behavior_reliability, dtype: float64

Reliability < 1.0: 181498
Reliability < 0.9: 1249
Reliability < 0.6: 13

Lowest reliability rows:
         user_id  ms_first_response  overlap_time  bottom_hint_missing  \
304309    178149              -6647         -6647                    0   
5806283   178149              -6208         -6208                    0   
5710459   178149              -5876         -5876                    0   
4634110   178149              -3210         -3210                    0   
4183441   209893              -2696         -2696                    0   
2805790   181809              -6767         -6767                    0   
2481278   178149              -6014         -6014                    0   
1943273   178149              -3607         -3607                    0   
1847590   178149

In [17]:
# =========================================================
# LEARNER-SPECIFIC RECENT BASELINE
# =========================================================

# Convert time
behavior_rel_df["start_time"] = pd.to_datetime(
    behavior_rel_df["start_time"],
    errors="coerce"
)

print(
    "Invalid start_time:",
    behavior_rel_df["start_time"].isna().sum()
)

# Sort each learner chronologically
behavior_rel_df = behavior_rel_df.sort_values(
    ["user_id", "start_time"]
).reset_index(drop=True)

# Invalid negative response times should not influence baseline
behavior_rel_df["rt_for_baseline"] = (
    behavior_rel_df["ms_first_response"]
    .where(
        behavior_rel_df["ms_first_response"] >= 0,
        np.nan
    )
)

# Response time is highly skewed -> log transform
behavior_rel_df["log_rt"] = np.log1p(
    behavior_rel_df["rt_for_baseline"]
)

# Number of previous interactions for this learner
behavior_rel_df["previous_interactions"] = (
    behavior_rel_df
    .groupby("user_id")
    .cumcount()
)

print("Sorted and prepared ✅")

print(
    behavior_rel_df[
        [
            "user_id",
            "start_time",
            "previous_interactions",
            "ms_first_response",
            "log_rt"
        ]
    ].head(10)
)

Invalid start_time: 73588
Sorted and prepared ✅
   user_id          start_time  previous_interactions  ms_first_response  \
0    21421 2013-05-22 14:44:20                      0              17482   
1    21421 2013-05-22 14:57:15                      1              25738   
2    21421 2013-05-22 14:58:06                      2              23005   
3    21421 2013-05-22 14:58:30                      3              16512   
4    21421 2013-06-05 13:00:06                      4               7890   
5    21421 2013-06-05 13:00:14                      5               7634   
6    21421 2013-06-05 13:00:22                      6               6309   
7    21421 2013-06-05 13:00:39                      7               5256   
8    21421 2013-06-05 13:00:45                      8               4970   
9    21421 2013-06-05 13:00:50                      9               7169   

      log_rt  
0   9.768984  
1  10.155763  
2  10.043510  
3   9.711903  
4   8.973478  
5   8.940498  
6   8.7498

In [18]:
WINDOW = 20
MIN_HISTORY = 5

g = behavior_rel_df.groupby(
    "user_id",
    sort=False
)

print("Creating recent learner baselines...")

behavior_rel_df["recent_rt_baseline"] = (
    g["log_rt"]
    .transform(
        lambda s:
        s.shift(1)
         .rolling(
             WINDOW,
             min_periods=MIN_HISTORY
         )
         .mean()
    )
)

behavior_rel_df["recent_attempt_baseline"] = (
    g["attempt_count"]
    .transform(
        lambda s:
        s.shift(1)
         .rolling(
             WINDOW,
             min_periods=MIN_HISTORY
         )
         .mean()
    )
)

behavior_rel_df["recent_hint_baseline"] = (
    g["hint_count"]
    .transform(
        lambda s:
        s.shift(1)
         .rolling(
             WINDOW,
             min_periods=MIN_HISTORY
         )
         .mean()
    )
)

behavior_rel_df["recent_accuracy_baseline"] = (
    g["correct"]
    .transform(
        lambda s:
        s.shift(1)
         .rolling(
             WINDOW,
             min_periods=MIN_HISTORY
         )
         .mean()
    )
)

print("Recent learner baselines created ✅")

Creating recent learner baselines...
Recent learner baselines created ✅


In [19]:
behavior_rel_df["rt_deviation"] = (
    behavior_rel_df["log_rt"]
    -
    behavior_rel_df["recent_rt_baseline"]
)

behavior_rel_df["attempt_deviation"] = (
    behavior_rel_df["attempt_count"]
    -
    behavior_rel_df["recent_attempt_baseline"]
)

behavior_rel_df["hint_deviation"] = (
    behavior_rel_df["hint_count"]
    -
    behavior_rel_df["recent_hint_baseline"]
)

behavior_rel_df["accuracy_deviation"] = (
    behavior_rel_df["correct"]
    -
    behavior_rel_df["recent_accuracy_baseline"]
)

In [20]:
baseline_cols = [
    "recent_rt_baseline",
    "recent_attempt_baseline",
    "recent_hint_baseline",
    "recent_accuracy_baseline"
]

print("=== BASELINE AVAILABILITY ===")

print(
    behavior_rel_df[
        baseline_cols
    ].isna().sum()
)

has_baseline = (
    behavior_rel_df[
        baseline_cols
    ].notna().all(axis=1)
)

print(
    "\nRows with learner baseline:",
    has_baseline.sum()
)

print(
    "Percentage with baseline:",
    round(
        has_baseline.mean() * 100,
        2
    ),
    "%"
)

print(
    "\nPrevious interaction statistics:"
)

print(
    behavior_rel_df[
        "previous_interactions"
    ].describe()
)

=== BASELINE AVAILABILITY ===
recent_rt_baseline          225327
recent_attempt_baseline     225327
recent_hint_baseline        225327
recent_accuracy_baseline    225327
dtype: int64

Rows with learner baseline: 5897943
Percentage with baseline: 96.32 %

Previous interaction statistics:
count    6.123270e+06
mean     2.709854e+02
std      3.597803e+02
min      0.000000e+00
25%      4.600000e+01
50%      1.400000e+02
75%      3.460000e+02
max      5.561000e+03
Name: previous_interactions, dtype: float64


In [21]:
# =========================================================
# HISTORY CONFIDENCE
# =========================================================

behavior_rel_df["history_confidence"] = np.clip(
    behavior_rel_df["previous_interactions"] / 20.0,
    0,
    1
)

# Baseline is considered usable only after 5 previous interactions
behavior_rel_df["baseline_available"] = (
    behavior_rel_df["previous_interactions"] >= 5
).astype(int)

print(
    behavior_rel_df[
        [
            "previous_interactions",
            "history_confidence",
            "baseline_available"
        ]
    ].head(25)
)

    previous_interactions  history_confidence  baseline_available
0                       0                0.00                   0
1                       1                0.05                   0
2                       2                0.10                   0
3                       3                0.15                   0
4                       4                0.20                   0
5                       5                0.25                   1
6                       6                0.30                   1
7                       7                0.35                   1
8                       8                0.40                   1
9                       9                0.45                   1
10                      0                0.00                   0
11                      1                0.05                   0
12                      2                0.10                   0
13                      3                0.15                   0
14        

In [22]:
behavior_rel_df["behavior_reliability_v2"] = (
    behavior_rel_df["behavior_reliability"]
    *
    (
        0.70
        +
        0.30
        * behavior_rel_df["history_confidence"]
    )
)

behavior_rel_df["behavior_reliability_v2"] = np.clip(
    behavior_rel_df["behavior_reliability_v2"],
    0,
    1
)

print(
    behavior_rel_df[
        "behavior_reliability_v2"
    ].describe()
)

print(
    "\nReliability v2 < 0.8:",
    (
        behavior_rel_df[
            "behavior_reliability_v2"
        ] < 0.8
    ).sum()
)

count    6.123270e+06
mean     9.756838e-01
std      6.554507e-02
min      4.000000e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: behavior_reliability_v2, dtype: float64

Reliability v2 < 0.8: 313612


In [23]:
behavior_rel_df["positive_rt_dev"] = np.maximum(
    behavior_rel_df["rt_deviation"],
    0
)

behavior_rel_df["positive_attempt_dev"] = np.maximum(
    behavior_rel_df["attempt_deviation"],
    0
)

behavior_rel_df["positive_hint_dev"] = np.maximum(
    behavior_rel_df["hint_deviation"],
    0
)

behavior_rel_df["accuracy_drop"] = np.maximum(
    -behavior_rel_df["accuracy_deviation"],
    0
)

In [24]:
personal_cols = [
    "positive_rt_dev",
    "positive_attempt_dev",
    "positive_hint_dev",
    "accuracy_drop"
]

personal_scales = {}

baseline_rows = behavior_rel_df[
    behavior_rel_df["baseline_available"] == 1
]

for col in personal_cols:

    scale = baseline_rows[col].quantile(0.95)

    personal_scales[col] = scale

    print(
        col,
        "95th percentile =",
        round(scale, 4)
    )

positive_rt_dev 95th percentile = 1.8177
positive_attempt_dev 95th percentile = 1.6
positive_hint_dev 95th percentile = 2.05
accuracy_drop 95th percentile = 0.8


In [25]:
from sklearn.model_selection import train_test_split
import numpy as np

# IMPORTANT:
# use original behavior_df order, before chronological sorting
all_users = behavior_df["user_id"].drop_duplicates().to_numpy()

train_users, temp_users = train_test_split(
    all_users,
    test_size=0.30,
    random_state=42
)

val_users, test_users = train_test_split(
    temp_users,
    test_size=0.50,
    random_state=42
)

print("Total learners:", len(all_users))
print("Train learners:", len(train_users))
print("Validation learners:", len(val_users))
print("Test learners:", len(test_users))

print(
    "Overlap train/val:",
    len(set(train_users) & set(val_users))
)

print(
    "Overlap train/test:",
    len(set(train_users) & set(test_users))
)

print(
    "Overlap val/test:",
    len(set(val_users) & set(test_users))
)

Total learners: 46674
Train learners: 32671
Validation learners: 7001
Test learners: 7002
Overlap train/val: 0
Overlap train/test: 0
Overlap val/test: 0


In [26]:
train_user_set = set(train_users)
val_user_set = set(val_users)

behavior_rel_df["split"] = np.where(
    behavior_rel_df["user_id"].isin(train_user_set),
    "train",
    np.where(
        behavior_rel_df["user_id"].isin(val_user_set),
        "validation",
        "test"
    )
)

print(
    behavior_rel_df["split"].value_counts()
)

split
train         4263157
test           935093
validation     925020
Name: count, dtype: int64


In [27]:
train_mask = (
    behavior_rel_df["split"] == "train"
)

valid_train_rt = behavior_rel_df.loc[
    train_mask
    & (behavior_rel_df["ms_first_response"] >= 0),
    "ms_first_response"
]

RT_LOW = valid_train_rt.quantile(0.01)
RT_HIGH = valid_train_rt.quantile(0.99)

print(
    "Train-only 1st percentile:",
    round(RT_LOW / 1000, 2),
    "seconds"
)

print(
    "Train-only 99th percentile:",
    round(RT_HIGH / 1000, 2),
    "seconds"
)

Train-only 1st percentile: 1.97 seconds
Train-only 99th percentile: 377.33 seconds


In [28]:
def add_behavior_reliability_train_based(df):

    result = df.copy()

    reliability = np.ones(
        len(result),
        dtype=float
    )

    # Missing bottom_hint
    reliability -= (
        result["bottom_hint_missing"].values
        * 0.10
    )

    # Invalid timing
    invalid_time = (
        (result["ms_first_response"] < 0)
        |
        (result["overlap_time"] < 0)
    )

    reliability -= (
        invalid_time.astype(float).values
        * 0.50
    )

    # Train-derived extreme short/long times
    very_short = (
        result["ms_first_response"] < RT_LOW
    )

    very_long = (
        result["ms_first_response"] > RT_HIGH
    )

    reliability -= (
        very_short.astype(float).values
        * 0.10
    )

    reliability -= (
        very_long.astype(float).values
        * 0.10
    )

    result["behavior_reliability"] = np.clip(
        reliability,
        0,
        1
    )

    return result


behavior_rel_df = (
    add_behavior_reliability_train_based(
        behavior_rel_df
    )
)

# Recreate v2
behavior_rel_df["behavior_reliability_v2"] = (
    behavior_rel_df["behavior_reliability"]
    *
    (
        0.70
        +
        0.30
        * behavior_rel_df["history_confidence"]
    )
)

behavior_rel_df["behavior_reliability_v2"] = np.clip(
    behavior_rel_df["behavior_reliability_v2"],
    0,
    1
)

In [29]:
train_baseline_rows = behavior_rel_df[
    (behavior_rel_df["split"] == "train")
    &
    (behavior_rel_df["baseline_available"] == 1)
]

personal_cols = [
    "positive_rt_dev",
    "positive_attempt_dev",
    "positive_hint_dev",
    "accuracy_drop"
]

personal_scales = {}

print("=== TRAIN-ONLY PERSONAL SCALES ===")

for col in personal_cols:

    scale = train_baseline_rows[
        col
    ].quantile(0.95)

    personal_scales[col] = scale

    print(
        col,
        "95th percentile =",
        round(scale, 4)
    )

=== TRAIN-ONLY PERSONAL SCALES ===
positive_rt_dev 95th percentile = 1.8174
positive_attempt_dev 95th percentile = 1.6
positive_hint_dev 95th percentile = 2.0667
accuracy_drop 95th percentile = 0.8


In [30]:
# =========================================================
# PERSONALIZED BEHAVIOUR DEVIATION SCORE
# =========================================================

for col in personal_cols:

    scale = personal_scales[col]

    behavior_rel_df[
        f"{col}_scaled"
    ] = np.clip(
        behavior_rel_df[col] / (scale + 1e-8),
        0,
        1
    )


scaled_cols = [
    "positive_rt_dev_scaled",
    "positive_attempt_dev_scaled",
    "positive_hint_dev_scaled",
    "accuracy_drop_scaled"
]


behavior_rel_df[
    "personalized_deviation_score"
] = behavior_rel_df[
    scaled_cols
].mean(axis=1)


# No usable baseline -> score should not be treated as evidence
behavior_rel_df.loc[
    behavior_rel_df["baseline_available"] == 0,
    "personalized_deviation_score"
] = np.nan


print(
    behavior_rel_df[
        "personalized_deviation_score"
    ].describe()
)

print(
    "\nRows with personalized score:",
    behavior_rel_df[
        "personalized_deviation_score"
    ].notna().sum()
)

count    5.897943e+06
mean     1.599104e-01
std      2.068141e-01
min      0.000000e+00
25%      0.000000e+00
50%      6.616779e-02
75%      2.500000e-01
max      1.000000e+00
Name: personalized_deviation_score, dtype: float64

Rows with personalized score: 5897943


In [31]:
behavior_rel_df[
    "personalized_behavior_evidence"
] = 0.5

has_history = (
    behavior_rel_df["baseline_available"] == 1
)

behavior_rel_df.loc[
    has_history,
    "personalized_behavior_evidence"
] = (
    0.5
    +
    behavior_rel_df.loc[
        has_history,
        "history_confidence"
    ]
    *
    (
        behavior_rel_df.loc[
            has_history,
            "personalized_deviation_score"
        ]
        - 0.5
    )
)

behavior_rel_df[
    "personalized_behavior_evidence"
] = np.clip(
    behavior_rel_df[
        "personalized_behavior_evidence"
    ],
    0,
    1
)

print(
    behavior_rel_df[
        "personalized_behavior_evidence"
    ].describe()
)

count    6.123270e+06
mean     1.855953e-01
std      2.116274e-01
min      0.000000e+00
25%      0.000000e+00
50%      1.065195e-01
75%      3.206833e-01
max      1.000000e+00
Name: personalized_behavior_evidence, dtype: float64


In [32]:
import glob
from xgboost import XGBClassifier

behavior_model_path = glob.glob(
    "/kaggle/input/**/mindbridge_behavior_model.json",
    recursive=True
)[0]

behavior_model = XGBClassifier()
behavior_model.load_model(
    behavior_model_path
)

print("Behaviour model loaded ✅")

Behaviour model loaded ✅


In [33]:
train_bottom_hint_median = (
    behavior_rel_df.loc[
        behavior_rel_df["split"] == "train",
        "bottom_hint"
    ].median()
)

print(
    "Train bottom_hint median:",
    train_bottom_hint_median
)

Train bottom_hint median: 0.0


In [34]:
behavior_features = [
    "correct",
    "bottom_hint",
    "hint_count",
    "attempt_count",
    "ms_first_response",
    "overlap_time",
    "first_action",
    "position",
    "original",
    "bottom_hint_missing"
]

CHUNK_SIZE = 250_000

behavior_prob = np.empty(
    len(behavior_rel_df),
    dtype=np.float32
)

for start in range(
    0,
    len(behavior_rel_df),
    CHUNK_SIZE
):

    end = min(
        start + CHUNK_SIZE,
        len(behavior_rel_df)
    )

    X_chunk = behavior_rel_df.iloc[
        start:end
    ][behavior_features].copy()

    X_chunk["bottom_hint"] = (
        X_chunk["bottom_hint"]
        .fillna(train_bottom_hint_median)
    )

    behavior_prob[start:end] = (
        behavior_model.predict_proba(
            X_chunk
        )[:, 1]
    )

    print(
        f"{end}/{len(behavior_rel_df)} predicted"
    )


behavior_rel_df[
    "behavior_confusion_prob"
] = behavior_prob

print("Behaviour probabilities created ✅")

250000/6123270 predicted
500000/6123270 predicted
750000/6123270 predicted
1000000/6123270 predicted
1250000/6123270 predicted
1500000/6123270 predicted
1750000/6123270 predicted
2000000/6123270 predicted
2250000/6123270 predicted
2500000/6123270 predicted
2750000/6123270 predicted
3000000/6123270 predicted
3250000/6123270 predicted
3500000/6123270 predicted
3750000/6123270 predicted
4000000/6123270 predicted
4250000/6123270 predicted
4500000/6123270 predicted
4750000/6123270 predicted
5000000/6123270 predicted
5250000/6123270 predicted
5500000/6123270 predicted
5750000/6123270 predicted
6000000/6123270 predicted
6123270/6123270 predicted
Behaviour probabilities created ✅


In [35]:
behavior_rel_df[
    "behavior_confusion_evidence"
] = (
    0.5
    +
    behavior_rel_df[
        "behavior_reliability_v2"
    ]
    *
    (
        behavior_rel_df[
            "behavior_confusion_prob"
        ]
        - 0.5
    )
)

behavior_rel_df[
    "behavior_confusion_evidence"
] = np.clip(
    behavior_rel_df[
        "behavior_confusion_evidence"
    ],
    0,
    1
)

print(
    behavior_rel_df[
        [
            "behavior_confusion_prob",
            "behavior_reliability_v2",
            "behavior_confusion_evidence",
            "personalized_behavior_evidence"
        ]
    ].describe()
)

       behavior_confusion_prob  behavior_reliability_v2  \
count             6.123270e+06             6.123270e+06   
mean              4.003073e-01             9.756823e-01   
std               2.158103e-01             6.554622e-02   
min               1.130284e-03             4.000000e-01   
25%               2.640000e-01             1.000000e+00   
50%               4.132163e-01             1.000000e+00   
75%               5.318562e-01             1.000000e+00   
max               9.296407e-01             1.000000e+00   

       behavior_confusion_evidence  personalized_behavior_evidence  
count                 6.123270e+06                    6.123270e+06  
mean                  4.029776e-01                    1.855953e-01  
std                   2.115313e-01                    2.116274e-01  
min                   1.219508e-03                    0.000000e+00  
25%                   2.709852e-01                    0.000000e+00  
50%                   4.152240e-01                    

In [36]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# Proxy confusion label used by Behaviour Model B
behavior_rel_df["confused_label"] = (
    behavior_rel_df[
        "Average_confidence(CONFUSED)"
    ] >= 0.5
).astype(int)


# Only rows where learner baseline exists
train_personal = behavior_rel_df[
    (behavior_rel_df["split"] == "train")
    &
    (behavior_rel_df["baseline_available"] == 1)
].copy()

val_personal = behavior_rel_df[
    (behavior_rel_df["split"] == "validation")
    &
    (behavior_rel_df["baseline_available"] == 1)
].copy()


# -------------------------------------------------
# Does personalized deviation actually predict
# confusion on unseen learners?
# -------------------------------------------------

val_auc = roc_auc_score(
    val_personal["confused_label"],
    val_personal["personalized_deviation_score"]
)

val_pr_auc = average_precision_score(
    val_personal["confused_label"],
    val_personal["personalized_deviation_score"]
)

print("=== PERSONALIZED DEVIATION VALIDATION ===")

print(
    "Validation ROC-AUC:",
    round(val_auc, 4)
)

print(
    "Validation PR-AUC:",
    round(val_pr_auc, 4)
)

print(
    "Validation confusion prevalence:",
    round(
        val_personal[
            "confused_label"
        ].mean(),
        4
    )
)

=== PERSONALIZED DEVIATION VALIDATION ===
Validation ROC-AUC: 0.3912
Validation PR-AUC: 0.0334
Validation confusion prevalence: 0.0421


In [37]:
# -------------------------------------------------
# Train-derived score bins
# -------------------------------------------------

_, bin_edges = pd.qcut(
    train_personal[
        "personalized_deviation_score"
    ],
    q=10,
    retbins=True,
    duplicates="drop"
)

bin_edges[0] = -np.inf
bin_edges[-1] = np.inf

val_personal["deviation_bin"] = pd.cut(
    val_personal[
        "personalized_deviation_score"
    ],
    bins=bin_edges,
    include_lowest=True
)

bin_summary = (
    val_personal
    .groupby(
        "deviation_bin",
        observed=True
    )
    .agg(
        rows=("confused_label", "size"),
        confusion_rate=("confused_label", "mean"),
        mean_score=(
            "personalized_deviation_score",
            "mean"
        )
    )
)

print("\n=== VALIDATION BY DEVIATION LEVEL ===")
print(bin_summary)


=== VALIDATION BY DEVIATION LEVEL ===
                    rows  confusion_rate  mean_score
deviation_bin                                       
(-inf, 0.0146]    356740        0.058213    0.000600
(0.0146, 0.0663]   89215        0.045968    0.039164
(0.0663, 0.133]    90035        0.038907    0.098452
(0.133, 0.219]     92473        0.034453    0.174514
(0.219, 0.313]     86935        0.031702    0.256623
(0.313, 0.484]     87739        0.024550    0.397907
(0.484, inf]       88120        0.011700    0.635031


In [38]:
inverse_score = (
    1.0
    - val_personal["personalized_deviation_score"]
)

inverse_auc = roc_auc_score(
    val_personal["confused_label"],
    inverse_score
)

inverse_pr_auc = average_precision_score(
    val_personal["confused_label"],
    inverse_score
)

print("=== INVERSE PERSONALIZED SCORE ===")
print("ROC-AUC:", round(inverse_auc, 4))
print("PR-AUC:", round(inverse_pr_auc, 4))

=== INVERSE PERSONALIZED SCORE ===
ROC-AUC: 0.6088
PR-AUC: 0.0557


In [39]:
for col in [
    "positive_rt_dev",
    "positive_attempt_dev",
    "positive_hint_dev",
    "accuracy_drop"
]:
    auc = roc_auc_score(
        val_personal["confused_label"],
        val_personal[col]
    )

    print(
        col,
        "ROC-AUC =",
        round(auc, 4)
    )

positive_rt_dev ROC-AUC = 0.423
positive_attempt_dev ROC-AUC = 0.4522
positive_hint_dev ROC-AUC = 0.455
accuracy_drop ROC-AUC = 0.4405


In [40]:
def reliability_aware_fusion(
    visual_prob,
    behavior_prob,
    visual_reliability,
    behavior_reliability
):
    """
    Reliability-aware multimodal fusion.

    Inputs must correspond to the SAME learner
    and SAME observation/time window.
    """

    visual_prob = np.asarray(
        visual_prob,
        dtype=float
    )

    behavior_prob = np.asarray(
        behavior_prob,
        dtype=float
    )

    visual_reliability = np.asarray(
        visual_reliability,
        dtype=float
    )

    behavior_reliability = np.asarray(
        behavior_reliability,
        dtype=float
    )

    # -----------------------------------
    # Reliability-based raw weights
    # -----------------------------------

    raw_visual_weight = np.clip(
        visual_reliability,
        0,
        1
    )

    raw_behavior_weight = np.clip(
        behavior_reliability,
        0,
        1
    )

    total_weight = (
        raw_visual_weight
        +
        raw_behavior_weight
    )

    # Avoid divide-by-zero
    total_weight = np.maximum(
        total_weight,
        1e-8
    )

    # -----------------------------------
    # Dynamic normalized weights
    # -----------------------------------

    visual_weight = (
        raw_visual_weight /
        total_weight
    )

    behavior_weight = (
        raw_behavior_weight /
        total_weight
    )

    # -----------------------------------
    # Reliability-aware fusion
    # -----------------------------------

    fused_probability = (
        visual_weight * visual_prob
        +
        behavior_weight * behavior_prob
    )

    return {
        "fused_probability":
            fused_probability,

        "visual_weight":
            visual_weight,

        "behavior_weight":
            behavior_weight
    }


print("Reliability-aware fusion function created ✅")

Reliability-aware fusion function created ✅


In [41]:
fusion_tests = pd.DataFrame({
    "case": [
        "Both reliable",
        "Visual unreliable",
        "Behaviour unreliable",
        "Both moderately reliable"
    ],

    "visual_prob": [
        0.80,
        0.90,
        0.80,
        0.70
    ],

    "behavior_prob": [
        0.70,
        0.30,
        0.20,
        0.60
    ],

    "visual_reliability": [
        0.95,
        0.20,
        0.95,
        0.60
    ],

    "behavior_reliability": [
        0.95,
        0.95,
        0.20,
        0.70
    ]
})


fusion_output = reliability_aware_fusion(
    fusion_tests["visual_prob"],
    fusion_tests["behavior_prob"],
    fusion_tests["visual_reliability"],
    fusion_tests["behavior_reliability"]
)


fusion_tests["visual_weight"] = (
    fusion_output["visual_weight"]
)

fusion_tests["behavior_weight"] = (
    fusion_output["behavior_weight"]
)

fusion_tests["fused_probability"] = (
    fusion_output["fused_probability"]
)

fusion_tests.round(4)

,case,visual_prob,behavior_prob,visual_reliability,behavior_reliability,visual_weight,behavior_weight,fused_probability
0,Both reliable,0.8,0.7,0.95,0.95,0.5000,0.5000,0.7500
1,Visual unreliable,0.9,0.3,0.20,0.95,0.1739,0.8261,0.4043
2,Behaviour unreliable,0.8,0.2,0.95,0.20,0.8261,0.1739,0.6957
3,Both moderately reliable,0.7,0.6,0.60,0.70,0.4615,0.5385,0.6462


In [42]:
def reliability_aware_fusion_v2(
    visual_prob,
    behavior_prob,
    visual_reliability,
    behavior_reliability
):
    """
    Reliability-aware multimodal fusion with
    modality disagreement handling.

    Inputs must represent the SAME learner
    and SAME observation/time window.
    """

    visual_prob = np.asarray(
        visual_prob,
        dtype=float
    )

    behavior_prob = np.asarray(
        behavior_prob,
        dtype=float
    )

    visual_reliability = np.clip(
        np.asarray(
            visual_reliability,
            dtype=float
        ),
        0,
        1
    )

    behavior_reliability = np.clip(
        np.asarray(
            behavior_reliability,
            dtype=float
        ),
        0,
        1
    )

    # ----------------------------------
    # 1. Dynamic reliability weights
    # ----------------------------------

    total_reliability = (
        visual_reliability
        +
        behavior_reliability
    )

    total_reliability = np.maximum(
        total_reliability,
        1e-8
    )

    visual_weight = (
        visual_reliability /
        total_reliability
    )

    behavior_weight = (
        behavior_reliability /
        total_reliability
    )

    # ----------------------------------
    # 2. Fused confusion probability
    # ----------------------------------

    fused_probability = (
        visual_weight * visual_prob
        +
        behavior_weight * behavior_prob
    )

    # ----------------------------------
    # 3. Modality disagreement
    # ----------------------------------

    disagreement = np.abs(
        visual_prob
        -
        behavior_prob
    )

    agreement = 1.0 - disagreement

    # ----------------------------------
    # 4. Reliability of dominant evidence
    # ----------------------------------

    weighted_reliability = (
        visual_weight * visual_reliability
        +
        behavior_weight * behavior_reliability
    )

    # ----------------------------------
    # 5. Conflict matters most when
    # BOTH modalities are reliable
    # ----------------------------------

    shared_reliability = np.minimum(
        visual_reliability,
        behavior_reliability
    )

    conflict_strength = (
        disagreement
        *
        shared_reliability
    )

    # ----------------------------------
    # 6. Final fusion confidence
    # ----------------------------------

    fusion_confidence = (
        weighted_reliability
        *
        (1.0 - conflict_strength)
    )

    fusion_confidence = np.clip(
        fusion_confidence,
        0,
        1
    )

    return {
        "fused_probability":
            fused_probability,

        "visual_weight":
            visual_weight,

        "behavior_weight":
            behavior_weight,

        "agreement":
            agreement,

        "disagreement":
            disagreement,

        "weighted_reliability":
            weighted_reliability,

        "conflict_strength":
            conflict_strength,

        "fusion_confidence":
            fusion_confidence
    }


print("Fusion v2 created ✅")

Fusion v2 created ✅


In [43]:
fusion_tests_v2 = pd.DataFrame({
    "case": [
        "Both agree and reliable",
        "Both disagree and reliable",
        "Visual unreliable conflict",
        "Behaviour unreliable conflict",
        "Both moderate and similar"
    ],

    "visual_prob": [
        0.80,
        0.90,
        0.90,
        0.80,
        0.70
    ],

    "behavior_prob": [
        0.75,
        0.10,
        0.10,
        0.20,
        0.60
    ],

    "visual_reliability": [
        0.95,
        0.95,
        0.20,
        0.95,
        0.60
    ],

    "behavior_reliability": [
        0.95,
        0.95,
        0.95,
        0.20,
        0.70
    ]
})


out = reliability_aware_fusion_v2(
    fusion_tests_v2["visual_prob"],
    fusion_tests_v2["behavior_prob"],
    fusion_tests_v2["visual_reliability"],
    fusion_tests_v2["behavior_reliability"]
)


for key in [
    "visual_weight",
    "behavior_weight",
    "fused_probability",
    "disagreement",
    "conflict_strength",
    "fusion_confidence"
]:
    fusion_tests_v2[key] = out[key]


fusion_tests_v2.round(4)

,case,visual_prob,behavior_prob,visual_reliability,behavior_reliability,visual_weight,behavior_weight,fused_probability,disagreement,conflict_strength,fusion_confidence
0,Both agree and reliable,0.8,0.75,0.95,0.95,0.5000,0.5000,0.7750,0.05,0.0475,0.9049
1,Both disagree and reliable,0.9,0.10,0.95,0.95,0.5000,0.5000,0.5000,0.80,0.7600,0.2280
2,Visual unreliable conflict,0.9,0.10,0.20,0.95,0.1739,0.8261,0.2391,0.80,0.1600,0.6884
3,Behaviour unreliable conflict,0.8,0.20,0.95,0.20,0.8261,0.1739,0.6957,0.60,0.1200,0.7212
4,Both moderate and similar,0.7,0.60,0.60,0.70,0.4615,0.5385,0.6462,0.10,0.0600,0.6146


In [44]:
def threshold_centered_evidence(prob, threshold):
    """
    Convert a modality's raw probability into a common
    0-1 evidence scale where its validated threshold = 0.5.

    0.0          -> 0.0
    threshold    -> 0.5
    1.0          -> 1.0
    """

    prob = np.asarray(prob, dtype=float)

    evidence = np.where(
        prob <= threshold,

        # Below threshold
        0.5 * (
            prob / (threshold + 1e-8)
        ),

        # Above threshold
        0.5 + 0.5 * (
            (prob - threshold) /
            (1.0 - threshold + 1e-8)
        )
    )

    return np.clip(
        evidence,
        0,
        1
    )


print("Threshold-centered evidence function ready ✅")

Threshold-centered evidence function ready ✅


In [45]:
test_probs = np.array([
    0.0,
    0.30,
    0.50,
    0.79,
    0.90,
    1.0
])

print("Visual threshold = 0.50")

print(
    np.round(
        threshold_centered_evidence(
            test_probs,
            0.50
        ),
        4
    )
)

print("\nBehaviour threshold = 0.79")

print(
    np.round(
        threshold_centered_evidence(
            test_probs,
            0.79
        ),
        4
    )
)

Visual threshold = 0.50
[0.   0.3  0.5  0.79 0.9  1.  ]

Behaviour threshold = 0.79
[0.     0.1899 0.3165 0.5    0.7619 1.    ]


In [46]:
train_visual_rel[
    "visual_normalized_evidence"
] = threshold_centered_evidence(
    train_visual_rel[
        "visual_confusion_prob"
    ],
    0.50
)

val_visual_rel[
    "visual_normalized_evidence"
] = threshold_centered_evidence(
    val_visual_rel[
        "visual_confusion_prob"
    ],
    0.50
)

test_visual_rel[
    "visual_normalized_evidence"
] = threshold_centered_evidence(
    test_visual_rel[
        "visual_confusion_prob"
    ],
    0.50
)

In [47]:
behavior_rel_df[
    "behavior_normalized_evidence"
] = threshold_centered_evidence(
    behavior_rel_df[
        "behavior_confusion_prob"
    ],
    0.79
)

In [48]:
train_visual_rel[
    "visual_final_evidence"
] = (
    0.5
    +
    train_visual_rel[
        "visual_reliability"
    ]
    *
    (
        train_visual_rel[
            "visual_normalized_evidence"
        ]
        - 0.5
    )
)


behavior_rel_df[
    "behavior_final_evidence"
] = (
    0.5
    +
    behavior_rel_df[
        "behavior_reliability_v2"
    ]
    *
    (
        behavior_rel_df[
            "behavior_normalized_evidence"
        ]
        - 0.5
    )
)

In [49]:
print("Visual:")
print(
    train_visual_rel[
        [
            "visual_confusion_prob",
            "visual_normalized_evidence",
            "visual_final_evidence"
        ]
    ].describe()
)

print("\nBehaviour:")
print(
    behavior_rel_df[
        [
            "behavior_confusion_prob",
            "behavior_normalized_evidence",
            "behavior_final_evidence"
        ]
    ].describe()
)

Visual:
       visual_confusion_prob  visual_normalized_evidence  \
count            5358.000000                 5358.000000   
mean                0.472470                    0.472470   
std                 0.141634                    0.141634   
min                 0.083300                    0.083300   
25%                 0.371132                    0.371132   
50%                 0.478182                    0.478182   
75%                 0.574321                    0.574321   
max                 0.889959                    0.889959   

       visual_final_evidence  
count            5358.000000  
mean                0.477380  
std                 0.130602  
min                 0.100542  
25%                 0.380083  
50%                 0.479427  
75%                 0.571099  
max                 0.889959  

Behaviour:
       behavior_confusion_prob  behavior_normalized_evidence  \
count             6.123270e+06                  6.123270e+06   
mean              4.003073e-01  

In [52]:
import numpy as np
import pandas as pd


def temporal_confusion_decision(
    fused_evidence,
    fusion_confidence,
    task_progress=None,
    window_size=3,
    confusion_threshold=0.50,
    intervention_threshold=0.60,
    min_persistent_windows=2,
    min_confidence=0.50
):
    """
    Temporal persistence analyser.

    fused_evidence:
        reliability-aware fused confusion evidence [0,1]

    fusion_confidence:
        confidence in multimodal fusion [0,1]

    task_progress:
        optional progress score [0,1]
        1 = learner progressing / succeeding
        0 = no progress

    Current thresholds are heuristic v1 and should
    later be calibrated using synchronized pilot data.
    """

    fused_evidence = np.asarray(
        fused_evidence,
        dtype=float
    )

    fusion_confidence = np.asarray(
        fusion_confidence,
        dtype=float
    )

    if task_progress is not None:
        task_progress = np.asarray(
            task_progress,
            dtype=float
        )

    results = []

    for i in range(len(fused_evidence)):

        start = max(
            0,
            i - window_size + 1
        )

        evidence_window = fused_evidence[
            start:i+1
        ]

        confidence_window = fusion_confidence[
            start:i+1
        ]

        # -----------------------------------
        # Confidence-weighted temporal score
        # -----------------------------------

        weight_sum = np.sum(
            confidence_window
        )

        if weight_sum > 0:

            temporal_score = np.sum(
                evidence_window
                * confidence_window
            ) / weight_sum

        else:
            temporal_score = 0.5

        avg_confidence = np.mean(
            confidence_window
        )

        # -----------------------------------
        # Persistent-confusion count
        # -----------------------------------

        valid_high = (
            (evidence_window >= confusion_threshold)
            &
            (confidence_window >= min_confidence)
        )

        persistent_count = int(
            np.sum(valid_high)
        )

        # -----------------------------------
        # Progress
        # -----------------------------------

        if task_progress is not None:

            progress_window = task_progress[
                start:i+1
            ]

            recent_progress = float(
                np.mean(progress_window)
            )

        else:
            recent_progress = np.nan

        # -----------------------------------
        # Decision logic - refined
        # -----------------------------------
        
        current_evidence = fused_evidence[i]
        
        if avg_confidence < 0.40:
        
            state = "uncertain"
            support_need = "collect_more_evidence"
        
        
        elif (
            current_evidence < confusion_threshold
            and
            temporal_score >= confusion_threshold
        ):
        
            # Previous windows were confused,
            # but the learner now appears to be recovering
            state = "recovering_from_confusion"
            support_need = "monitor"
        
        
        elif (
            current_evidence >= confusion_threshold
            and
            temporal_score >= intervention_threshold
            and
            persistent_count >= min_persistent_windows
        ):
        
            if (
                task_progress is not None
                and
                recent_progress >= 0.50
            ):
        
                state = "productive_confusion"
                support_need = "monitor"
        
            else:
        
                state = "intervention_worthy_confusion"
                support_need = "request_tutor_support"
        
        
        elif current_evidence >= confusion_threshold:
        
            state = "transient_confusion"
            support_need = "monitor"
        
        
        else:
        
            state = "no_confusion"
            support_need = "none"

        results.append({
            "temporal_score": temporal_score,
            "avg_confidence": avg_confidence,
            "persistent_count": persistent_count,
            "recent_progress": recent_progress,
            "state": state,
            "support_need": support_need
        })

    return pd.DataFrame(results)


print("Temporal persistence analyser created ✅")

Temporal persistence analyser created ✅


In [53]:
demo = pd.DataFrame({

    "fused_evidence": [
        0.30,
        0.52,
        0.63,
        0.70,
        0.72,
        0.40
    ],

    "fusion_confidence": [
        0.90,
        0.85,
        0.88,
        0.90,
        0.92,
        0.90
    ],

    # 1 = progressing
    # 0 = not progressing
    "task_progress": [
        1,
        1,
        1,
        0,
        0,
        1
    ]
})


temporal_output = temporal_confusion_decision(
    demo["fused_evidence"],
    demo["fusion_confidence"],
    demo["task_progress"]
)

demo_result = pd.concat(
    [
        demo,
        temporal_output
    ],
    axis=1
)

demo_result.round(4)

,fused_evidence,fusion_confidence,task_progress,temporal_score,avg_confidence,persistent_count,recent_progress,state,support_need
0,0.30,0.90,1,0.3000,0.9000,0,1.0000,no_confusion,none
1,0.52,0.85,1,0.4069,0.8750,1,1.0000,transient_confusion,monitor
2,0.63,0.88,1,0.4815,0.8767,2,1.0000,transient_confusion,monitor
3,0.70,0.90,0,0.6184,0.8767,3,0.6667,productive_confusion,monitor
4,0.72,0.92,0,0.6840,0.9000,3,0.3333,intervention_worthy_confusion,request_tutor_support
5,0.40,0.90,1,0.6075,0.9067,2,0.3333,recovering_from_confusion,monitor


In [54]:
def mindbridge_learner_state_engine(
    visual_evidence,
    behavior_evidence,
    visual_reliability,
    behavior_reliability,
    task_progress=None,
    window_size=3,
    confusion_threshold=0.50,
    intervention_threshold=0.60
):
    """
    MindBridge Reliability-Aware Learner-State Engine.

    IMPORTANT:
    Inputs must correspond to synchronized observations
    from the SAME learner and SAME time windows.

    visual_evidence:
        threshold-centered visual confusion evidence [0,1]

    behavior_evidence:
        threshold-centered behaviour confusion evidence [0,1]

    visual_reliability:
        visual modality reliability [0,1]

    behavior_reliability:
        behaviour modality reliability [0,1]

    task_progress:
        optional progress score [0,1]
    """

    # -----------------------------------------
    # 1. Reliability-aware multimodal fusion
    # -----------------------------------------

    fusion = reliability_aware_fusion_v2(
        visual_evidence,
        behavior_evidence,
        visual_reliability,
        behavior_reliability
    )

    # -----------------------------------------
    # 2. Temporal persistence analysis
    # -----------------------------------------

    temporal = temporal_confusion_decision(
        fused_evidence=fusion["fused_probability"],
        fusion_confidence=fusion["fusion_confidence"],
        task_progress=task_progress,
        window_size=window_size,
        confusion_threshold=confusion_threshold,
        intervention_threshold=intervention_threshold
    )

    # -----------------------------------------
    # 3. Final API-ready output
    # -----------------------------------------

    output = pd.DataFrame({
        "visual_evidence": np.asarray(
            visual_evidence,
            dtype=float
        ),

        "behavior_evidence": np.asarray(
            behavior_evidence,
            dtype=float
        ),

        "visual_reliability": np.asarray(
            visual_reliability,
            dtype=float
        ),

        "behavior_reliability": np.asarray(
            behavior_reliability,
            dtype=float
        ),

        "visual_weight":
            fusion["visual_weight"],

        "behavior_weight":
            fusion["behavior_weight"],

        "fused_evidence":
            fusion["fused_probability"],

        "modality_disagreement":
            fusion["disagreement"],

        "fusion_confidence":
            fusion["fusion_confidence"]
    })

    output = pd.concat(
        [
            output.reset_index(drop=True),
            temporal.reset_index(drop=True)
        ],
        axis=1
    )

    return output


print("MindBridge learner-state engine created ✅")

MindBridge learner-state engine created ✅


In [55]:
final_demo = mindbridge_learner_state_engine(

    visual_evidence=[
        0.30,
        0.55,
        0.65,
        0.72,
        0.78,
        0.40
    ],

    behavior_evidence=[
        0.35,
        0.50,
        0.60,
        0.68,
        0.75,
        0.42
    ],

    visual_reliability=[
        0.95,
        0.90,
        0.85,
        0.80,
        0.75,
        0.90
    ],

    behavior_reliability=[
        0.95,
        0.95,
        0.95,
        0.95,
        0.95,
        0.95
    ],

    task_progress=[
        1,
        1,
        1,
        0,
        0,
        1
    ]
)

final_demo.round(4)

,visual_evidence,behavior_evidence,visual_reliability,behavior_reliability,visual_weight,behavior_weight,fused_evidence,modality_disagreement,fusion_confidence,temporal_score,avg_confidence,persistent_count,recent_progress,state,support_need
0,0.30,0.35,0.95,0.95,0.5000,0.5000,0.3250,0.05,0.9049,0.3250,0.9049,0,1.0000,no_confusion,none
1,0.55,0.50,0.90,0.95,0.4865,0.5135,0.5243,0.05,0.8840,0.4235,0.8944,1,1.0000,transient_confusion,monitor
2,0.65,0.60,0.85,0.95,0.4722,0.5278,0.6236,0.05,0.8644,0.4887,0.8844,2,1.0000,transient_confusion,monitor
3,0.72,0.68,0.80,0.95,0.4571,0.5429,0.6983,0.04,0.8532,0.6144,0.8672,3,0.6667,productive_confusion,monitor
4,0.78,0.75,0.75,0.95,0.4412,0.5588,0.7632,0.03,0.8424,0.6944,0.8533,3,0.3333,intervention_worthy_confusion,request_tutor_support
5,0.40,0.42,0.90,0.95,0.4865,0.5135,0.4103,0.02,0.9090,0.6188,0.8682,2,0.3333,recovering_from_confusion,monitor


In [56]:
import json

fusion_config = {
    "system": "MindBridge Learner-State Engine",
    "version": "prototype_v1",

    "visual_model": {
        "threshold": 0.50,
        "dataset": "DAiSEE"
    },

    "behavior_model": {
        "threshold": 0.79,
        "dataset": "ASSISTments 2012-2013 with Affect"
    },

    "reliability": {
        "visual": "visibility + train-derived pose quality",
        "behavior": "data quality + history confidence"
    },

    "learner_history": {
        "window_size": 20,
        "minimum_history": 5,
        "note": (
            "Baseline deviations retained as contextual features. "
            "They are not hard-coded as positive confusion evidence "
            "because the public validation data did not support that direction."
        )
    },

    "fusion": {
        "method": "normalized reliability-weighted fusion",
        "conflict_handling": True,
        "confusion_threshold": 0.50
    },

    "temporal": {
        "window_size": 3,
        "intervention_threshold": 0.60,
        "minimum_persistent_windows": 2,
        "minimum_confidence": 0.50
    },

    "states": [
        "no_confusion",
        "transient_confusion",
        "productive_confusion",
        "intervention_worthy_confusion",
        "recovering_from_confusion",
        "uncertain"
    ],

    "support_actions": [
        "none",
        "monitor",
        "request_tutor_support",
        "collect_more_evidence"
    ]
}

config_path = (
    "/kaggle/working/"
    "mindbridge_fusion_config.json"
)

with open(config_path, "w") as f:
    json.dump(
        fusion_config,
        f,
        indent=4
    )

print("Fusion configuration saved ✅")
print(config_path)

Fusion configuration saved ✅
/kaggle/working/mindbridge_fusion_config.json


In [57]:
import pandas as pd

pilot_columns = [
    # Identity / timing
    "session_id",
    "learner_id",
    "window_id",
    "window_start",
    "window_end",

    # Visual branch
    "visual_confusion_prob",
    "visual_normalized_evidence",
    "visual_reliability",

    # Behaviour branch
    "behavior_confusion_prob",
    "behavior_normalized_evidence",
    "behavior_reliability",

    # Learner-specific context
    "history_confidence",
    "rt_deviation",
    "attempt_deviation",
    "hint_deviation",
    "accuracy_deviation",

    # Learning progress
    "task_progress",
    "current_correct",
    "recent_accuracy",

    # Fusion outputs
    "visual_weight",
    "behavior_weight",
    "fused_evidence",
    "modality_disagreement",
    "fusion_confidence",

    # Temporal / final decision
    "temporal_score",
    "persistent_count",
    "learner_state",
    "support_need",

    # Ground truth for research evaluation
    "self_report_confusion",
    "observer_confusion",
    "task_outcome"
]

pilot_template = pd.DataFrame(
    columns=pilot_columns
)

pilot_path = (
    "/kaggle/working/"
    "mindbridge_pilot_data_template.csv"
)

pilot_template.to_csv(
    pilot_path,
    index=False
)

print("Pilot data template created ✅")
print("Columns:", len(pilot_columns))
print("Saved:", pilot_path)

pilot_template

Pilot data template created ✅
Columns: 31
Saved: /kaggle/working/mindbridge_pilot_data_template.csv


,session_id,learner_id,window_id,window_start,window_end,visual_confusion_prob,visual_normalized_evidence,visual_reliability,behavior_confusion_prob,behavior_normalized_evidence,...,fused_evidence,modality_disagreement,fusion_confidence,temporal_score,persistent_count,learner_state,support_need,self_report_confusion,observer_confusion,task_outcome


In [58]:
# =========================================================
# EXAMPLE SYNCHRONIZED PILOT WINDOW
# Synthetic example only
# =========================================================

example_visual_prob = 0.72
example_visual_reliability = 0.82

example_behavior_prob = 0.85
example_behavior_reliability = 0.95

# -----------------------------------------
# Threshold-centered evidence
# -----------------------------------------

visual_ev = float(
    threshold_centered_evidence(
        [example_visual_prob],
        0.50
    )[0]
)

behavior_ev = float(
    threshold_centered_evidence(
        [example_behavior_prob],
        0.79
    )[0]
)

# -----------------------------------------
# Reliability-aware fusion
# -----------------------------------------

fusion = reliability_aware_fusion_v2(
    [visual_ev],
    [behavior_ev],
    [example_visual_reliability],
    [example_behavior_reliability]
)

# -----------------------------------------
# Temporal decision for this first window
# -----------------------------------------

temporal = temporal_confusion_decision(
    fused_evidence=fusion["fused_probability"],
    fusion_confidence=fusion["fusion_confidence"],
    task_progress=[0]
)

# -----------------------------------------
# Create one pilot row
# -----------------------------------------

example_row = {
    "session_id": "S001",
    "learner_id": "L001",
    "window_id": 1,

    "window_start": "2026-10-08 10:00:00",
    "window_end": "2026-10-08 10:00:20",

    # Visual
    "visual_confusion_prob": example_visual_prob,
    "visual_normalized_evidence": visual_ev,
    "visual_reliability": example_visual_reliability,

    # Behaviour
    "behavior_confusion_prob": example_behavior_prob,
    "behavior_normalized_evidence": behavior_ev,
    "behavior_reliability": example_behavior_reliability,

    # Learner history/context
    "history_confidence": 0.90,
    "rt_deviation": 0.35,
    "attempt_deviation": 1.0,
    "hint_deviation": 1.0,
    "accuracy_deviation": -0.40,

    # Progress
    "task_progress": 0,
    "current_correct": 0,
    "recent_accuracy": 0.60,

    # Fusion
    "visual_weight": float(
        fusion["visual_weight"][0]
    ),

    "behavior_weight": float(
        fusion["behavior_weight"][0]
    ),

    "fused_evidence": float(
        fusion["fused_probability"][0]
    ),

    "modality_disagreement": float(
        fusion["disagreement"][0]
    ),

    "fusion_confidence": float(
        fusion["fusion_confidence"][0]
    ),

    # Temporal
    "temporal_score": float(
        temporal.loc[0, "temporal_score"]
    ),

    "persistent_count": int(
        temporal.loc[0, "persistent_count"]
    ),

    "learner_state":
        temporal.loc[0, "state"],

    "support_need":
        temporal.loc[0, "support_need"],

    # Ground truth - collected separately
    "self_report_confusion": np.nan,
    "observer_confusion": np.nan,
    "task_outcome": "pending"
}

example_pilot_df = pd.DataFrame(
    [example_row],
    columns=pilot_columns
)

example_pilot_df.T

,0
session_id,S001
learner_id,L001
window_id,1
window_start,2026-10-08 10:00:00
window_end,2026-10-08 10:00:20
visual_confusion_prob,0.72
visual_normalized_evidence,0.72
visual_reliability,0.82
behavior_confusion_prob,0.85
behavior_normalized_evidence,0.642857


In [59]:
# =========================================================
# SYNTHETIC 3-WINDOW PILOT EXAMPLE
# SAME learner + SAME session
# =========================================================

pilot_sequence = pd.DataFrame({

    "window_id": [1, 2, 3],

    "visual_confusion_prob": [
        0.72,
        0.76,
        0.80
    ],

    "behavior_confusion_prob": [
        0.85,
        0.87,
        0.89
    ],

    "visual_reliability": [
        0.82,
        0.80,
        0.78
    ],

    "behavior_reliability": [
        0.95,
        0.95,
        0.95
    ],

    # Learner progress decreases
    "task_progress": [
        1,
        0,
        0
    ]
})


# -----------------------------------------
# Threshold normalization
# -----------------------------------------

pilot_sequence[
    "visual_evidence"
] = threshold_centered_evidence(
    pilot_sequence[
        "visual_confusion_prob"
    ],
    0.50
)

pilot_sequence[
    "behavior_evidence"
] = threshold_centered_evidence(
    pilot_sequence[
        "behavior_confusion_prob"
    ],
    0.79
)


# -----------------------------------------
# Reliability-aware fusion
# -----------------------------------------

fusion = reliability_aware_fusion_v2(
    pilot_sequence[
        "visual_evidence"
    ],

    pilot_sequence[
        "behavior_evidence"
    ],

    pilot_sequence[
        "visual_reliability"
    ],

    pilot_sequence[
        "behavior_reliability"
    ]
)


pilot_sequence[
    "visual_weight"
] = fusion["visual_weight"]

pilot_sequence[
    "behavior_weight"
] = fusion["behavior_weight"]

pilot_sequence[
    "fused_evidence"
] = fusion["fused_probability"]

pilot_sequence[
    "fusion_confidence"
] = fusion["fusion_confidence"]


# -----------------------------------------
# Temporal persistence
# -----------------------------------------

temporal = temporal_confusion_decision(

    fused_evidence=
        pilot_sequence["fused_evidence"],

    fusion_confidence=
        pilot_sequence["fusion_confidence"],

    task_progress=
        pilot_sequence["task_progress"],

    window_size=3,

    confusion_threshold=0.50,

    intervention_threshold=0.60
)


pilot_sequence[
    "temporal_score"
] = temporal["temporal_score"]

pilot_sequence[
    "persistent_count"
] = temporal["persistent_count"]

pilot_sequence[
    "recent_progress"
] = temporal["recent_progress"]

pilot_sequence[
    "learner_state"
] = temporal["state"]

pilot_sequence[
    "support_need"
] = temporal["support_need"]


pilot_sequence.round(4)

,window_id,visual_confusion_prob,behavior_confusion_prob,visual_reliability,behavior_reliability,task_progress,visual_evidence,behavior_evidence,visual_weight,behavior_weight,fused_evidence,fusion_confidence,temporal_score,persistent_count,recent_progress,learner_state,support_need
0,1,0.72,0.85,0.82,0.95,1,0.72,0.6429,0.4633,0.5367,0.6786,0.8335,0.6786,1,1.0000,transient_confusion,monitor
1,2,0.76,0.87,0.80,0.95,0,0.76,0.6905,0.4571,0.5429,0.7223,0.8324,0.7004,2,0.5000,productive_confusion,monitor
2,3,0.80,0.89,0.78,0.95,0,0.80,0.7381,0.4509,0.5491,0.7660,0.8312,0.7222,3,0.3333,intervention_worthy_confusion,request_tutor_support


In [60]:
from pydantic import BaseModel
from typing import Optional


class LearnerStateRequest(BaseModel):
    session_id: str
    learner_id: str
    window_id: int

    # Model outputs
    visual_confusion_prob: float
    behavior_confusion_prob: float

    # Reliability
    visual_reliability: float
    behavior_reliability: float

    # Learning progress
    task_progress: Optional[float] = None


class LearnerStateResponse(BaseModel):
    session_id: str
    learner_id: str
    window_id: int

    visual_weight: float
    behavior_weight: float

    fused_evidence: float
    modality_disagreement: float
    fusion_confidence: float

    temporal_score: float
    persistent_count: int

    learner_state: str
    support_need: str


print("API request/response schemas created ✅")

API request/response schemas created ✅


In [61]:
example_request = LearnerStateRequest(
    session_id="S001",
    learner_id="L001",
    window_id=1,

    visual_confusion_prob=0.72,
    behavior_confusion_prob=0.85,

    visual_reliability=0.82,
    behavior_reliability=0.95,

    task_progress=0.0
)

print(example_request.model_dump())

{'session_id': 'S001', 'learner_id': 'L001', 'window_id': 1, 'visual_confusion_prob': 0.72, 'behavior_confusion_prob': 0.85, 'visual_reliability': 0.82, 'behavior_reliability': 0.95, 'task_progress': 0.0}


In [62]:
# =========================================================
# REAL-TIME LEARNER STATE SERVICE LOGIC
# =========================================================

# Temporary in-memory history for prototype
# Later this can move to Redis / PostgreSQL.
session_history = {}


def reset_session_history():
    session_history.clear()
    print("Session history cleared ✅")


def process_learner_state_request(
    request: LearnerStateRequest
) -> LearnerStateResponse:

    # -----------------------------------------
    # 1. Threshold-center model probabilities
    # -----------------------------------------

    visual_evidence = float(
        threshold_centered_evidence(
            [request.visual_confusion_prob],
            0.50
        )[0]
    )

    behavior_evidence = float(
        threshold_centered_evidence(
            [request.behavior_confusion_prob],
            0.79
        )[0]
    )

    # -----------------------------------------
    # 2. Reliability-aware fusion
    # -----------------------------------------

    fusion = reliability_aware_fusion_v2(

        [visual_evidence],

        [behavior_evidence],

        [request.visual_reliability],

        [request.behavior_reliability]
    )

    fused_evidence = float(
        fusion["fused_probability"][0]
    )

    fusion_confidence = float(
        fusion["fusion_confidence"][0]
    )

    # -----------------------------------------
    # 3. Session key
    # -----------------------------------------

    key = (
        request.session_id,
        request.learner_id
    )

    if key not in session_history:

        session_history[key] = {
            "fused_evidence": [],
            "fusion_confidence": [],
            "task_progress": []
        }

    history = session_history[key]

    # -----------------------------------------
    # 4. Save current window
    # -----------------------------------------

    history["fused_evidence"].append(
        fused_evidence
    )

    history["fusion_confidence"].append(
        fusion_confidence
    )

    history["task_progress"].append(
        request.task_progress
    )

    # Keep only recent history
    history["fused_evidence"] = (
        history["fused_evidence"][-10:]
    )

    history["fusion_confidence"] = (
        history["fusion_confidence"][-10:]
    )

    history["task_progress"] = (
        history["task_progress"][-10:]
    )

    # -----------------------------------------
    # 5. Temporal analysis
    # -----------------------------------------

    # For current prototype, task_progress should
    # normally be supplied for each pilot window.
    progress_available = all(
        value is not None
        for value in history["task_progress"]
    )

    if progress_available:

        progress_input = history[
            "task_progress"
        ]

    else:

        progress_input = None

    temporal = temporal_confusion_decision(

        fused_evidence=
            history["fused_evidence"],

        fusion_confidence=
            history["fusion_confidence"],

        task_progress=
            progress_input,

        window_size=3,

        confusion_threshold=0.50,

        intervention_threshold=0.60,

        min_persistent_windows=2,

        min_confidence=0.50
    )

    latest = temporal.iloc[-1]

    # -----------------------------------------
    # 6. API-ready response
    # -----------------------------------------

    response = LearnerStateResponse(

        session_id=request.session_id,
        learner_id=request.learner_id,
        window_id=request.window_id,

        visual_weight=float(
            fusion["visual_weight"][0]
        ),

        behavior_weight=float(
            fusion["behavior_weight"][0]
        ),

        fused_evidence=fused_evidence,

        modality_disagreement=float(
            fusion["disagreement"][0]
        ),

        fusion_confidence=
            fusion_confidence,

        temporal_score=float(
            latest["temporal_score"]
        ),

        persistent_count=int(
            latest["persistent_count"]
        ),

        learner_state=str(
            latest["state"]
        ),

        support_need=str(
            latest["support_need"]
        )
    )

    return response


print("MindBridge service logic ready ✅")

MindBridge service logic ready ✅


In [63]:
reset_session_history()

response_1 = process_learner_state_request(
    example_request
)

print(
    response_1.model_dump()
)

Session history cleared ✅
{'session_id': 'S001', 'learner_id': 'L001', 'window_id': 1, 'visual_weight': 0.46327683615819204, 'behavior_weight': 0.5367231638418078, 'fused_evidence': 0.6785956359568931, 'modality_disagreement': 0.07714285954557809, 'fusion_confidence': 0.833489447803033, 'temporal_score': 0.6785956359568931, 'persistent_count': 1, 'learner_state': 'transient_confusion', 'support_need': 'monitor'}


In [64]:
request_2 = LearnerStateRequest(
    session_id="S001",
    learner_id="L001",
    window_id=2,

    visual_confusion_prob=0.76,
    behavior_confusion_prob=0.87,

    visual_reliability=0.80,
    behavior_reliability=0.95,

    task_progress=0.0
)

response_2 = process_learner_state_request(
    request_2
)

print("WINDOW 2")
print(response_2.model_dump())

WINDOW 2
{'session_id': 'S001', 'learner_id': 'L001', 'window_id': 2, 'visual_weight': 0.4571428571428572, 'behavior_weight': 0.5428571428571428, 'fused_evidence': 0.7222584961003435, 'modality_disagreement': 0.06952381339410396, 'fusion_confidence': 0.8324043510123862, 'temporal_score': 0.7004128459056291, 'persistent_count': 2, 'learner_state': 'intervention_worthy_confusion', 'support_need': 'request_tutor_support'}


In [66]:
request_3 = LearnerStateRequest(
    session_id="S001",
    learner_id="L001",
    window_id=3,

    visual_confusion_prob=0.80,
    behavior_confusion_prob=0.89,

    visual_reliability=0.78,
    behavior_reliability=0.95,

    task_progress=0.0
)

response_3 = process_learner_state_request(
    request_3
)

print("WINDOW 3")
print(response_3.model_dump())

WINDOW 3
{'session_id': 'S001', 'learner_id': 'L001', 'window_id': 3, 'visual_weight': 0.4508670520231214, 'behavior_weight': 0.5491329479768786, 'fused_evidence': 0.7660060466702323, 'modality_disagreement': 0.061904767242630054, 'fusion_confidence': 0.8311821433497261, 'temporal_score': 0.7514092416238495, 'persistent_count': 3, 'learner_state': 'intervention_worthy_confusion', 'support_need': 'request_tutor_support'}


In [67]:
import json

confusion_engine_summary = {
    "component": "MindBridge Reliability-Aware Confusion Detection",
    "status": "prototype_complete",

    "visual_model": {
        "model": "XGBoost",
        "dataset": "DAiSEE",
        "threshold": 0.50,
        "test_metrics": {
            "roc_auc": 0.5757,
            "pr_auc": 0.3883,
            "macro_f1": 0.5426,
            "confused_precision": 0.3832,
            "confused_recall": 0.5788,
            "confused_f1": 0.4611
        }
    },

    "behavior_model": {
        "model": "XGBoost",
        "dataset": "ASSISTments 2012-2013 with Affect",
        "threshold": 0.79,
        "test_metrics": {
            "roc_auc": 0.7604,
            "pr_auc": 0.1444,
            "macro_f1": 0.5922,
            "confused_precision": 0.1822,
            "confused_recall": 0.3173,
            "confused_f1": 0.2315
        }
    },

    "learner_baseline": {
        "window": 20,
        "minimum_history": 5,
        "coverage_percent": 96.32,
        "deviation_validation_roc_auc": 0.3912,
        "inverse_deviation_roc_auc": 0.6088,
        "decision": (
            "Retain learner-specific deviations as contextual features; "
            "do not hard-code larger deviation as positive confusion evidence."
        )
    },

    "fusion": {
        "method": "dynamic reliability-weighted multimodal fusion",
        "threshold_centering": True,
        "modality_conflict_handling": True,
        "temporal_window": 3,
        "confusion_threshold": 0.50,
        "intervention_threshold": 0.60,
        "minimum_persistent_windows": 2
    },

    "states": [
        "no_confusion",
        "transient_confusion",
        "productive_confusion",
        "intervention_worthy_confusion",
        "recovering_from_confusion",
        "uncertain"
    ],

    "final_note": (
        "Core confusion-detection prototype is complete. "
        "Final multimodal validation requires synchronized MindBridge pilot data."
    )
}

path = "/kaggle/working/mindbridge_confusion_engine_summary.json"

with open(path, "w") as f:
    json.dump(
        confusion_engine_summary,
        f,
        indent=4
    )

print("Confusion engine summary saved ✅")
print(path)

Confusion engine summary saved ✅
/kaggle/working/mindbridge_confusion_engine_summary.json
